# 01 · RxRx3-core 数据探索与扰动图像对比

只读元数据、原图和预计算 embedding 的文件信息，不训练模型、不改写数据集。
对比 CRISPR、药物浓度和匹配阴性对照，也可直接指定任意 well。
“阴性对照”按明确规则选择，不代表细胞完全没有接受任何处理；intron 背景、溶剂等仍可能存在。

In [ ]:
import json
import os
import sys
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from PIL import Image

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from phenostate.data.source import load_metadata
from phenostate.data.exploration import read_selected_wells
from phenostate.embedding_metrics import evaluation_metadata

DATA_ROOT = Path(os.environ.get('RXRX3_CORE_ROOT', '/home/xkj2006/CellPainting/rxrx3_core')).expanduser()
CONTROL_RULES_PATH = PROJECT_ROOT / 'configs/negative_controls.example.json'  # 先核实实际对照定义。
MDS_ROOT_TEXT = os.environ.get('RXRX3_MDS_ROOT', '')
metadata_path = DATA_ROOT / 'metadata_rxrx3_core.csv'
metadata = pd.DataFrame(evaluation_metadata(list(load_metadata(metadata_path).values())))
metadata['concentration_um'] = pd.to_numeric(metadata['concentration'], errors='coerce')
shards = sorted((DATA_ROOT / 'data').glob('train-*.parquet'))
print('Data:', DATA_ROOT, '| image shards:', len(shards))

## 1. 元数据、重复与浓度分布

In [ ]:
display(pd.Series({'wells': len(metadata), 'unique_wells': metadata.well_id.nunique(),
                   'experiments': metadata.experiment_name.nunique(),
                   'plates': len(metadata[['experiment_name', 'plate']].drop_duplicates())}, name='count').to_frame())
display(metadata.head())
display(metadata['well_type_label'].value_counts(dropna=False).to_frame('wells'))
display(metadata.loc[metadata.is_query].groupby(['perturbation_type', 'treatment', 'concentration'],
                                               dropna=False).size().rename('replicate_wells').describe().to_frame())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metadata.perturbation_type.value_counts().plot.bar(ax=axes[0], title='Perturbation types', rot=0)
metadata.loc[metadata.perturbation_type == 'COMPOUND', 'concentration_um'].value_counts().sort_index().plot.bar(
    ax=axes[1], title='Compound concentrations (uM)', rot=45)
for ax in axes:
    ax.set_ylabel('Wells')
plt.tight_layout()
plt.show()

## 2. 图像和公开 embedding 文件概况

只读 Parquet schema 和行数，展示文件的实际维度、孔位数。三种 embedding 已经是 well 级特征；没有图像 decoder，不能用它们画重建图。

In [ ]:
PUBLIC_FILES = {
    'OpenPhenom': 'OpenPhenom_rxrx3_core_embeddings.parquet',
    'MAE-L-8': 'MAE-L_8_RPI-93M_rxrx3_core_embeddings.parquet',
    'MAE-G-8': 'MAE-G_8_PP-16M_trimmed_rxrx3_core_embeddings.parquet',
}
inventory = []
for name, filename in PUBLIC_FILES.items():
    path = DATA_ROOT / filename
    if not path.is_file():
        inventory.append({'model': name, 'file': filename, 'status': 'missing'})
        continue
    parquet = pq.ParquetFile(path)
    schema = parquet.schema_arrow
    if 'embedding' in schema.names:
        dtype = schema.field('embedding').type
        dimension = dtype.list_size if pa.types.is_fixed_size_list(dtype) else 'variable list; inspect vectors'
    else:
        dimension = sum(c.startswith('feature_') and c[8:].isdigit() for c in schema.names)
    inventory.append({'model': name, 'file': filename, 'wells': parquet.metadata.num_rows,
                      'dimension': dimension, 'size_GB': round(path.stat().st_size / 1e9, 3), 'status': 'present'})
display(pd.DataFrame(inventory))
if shards:
    image_rows = sum(pq.ParquetFile(p).metadata.num_rows for p in shards)
    print('Single-channel image rows:', image_rows, '| metadata wells x 6:', len(metadata) * 6)

## 3. 选择对比对象

直接填写 `WELL_IDS` 可查看任意孔位；留空时自动展示几组 CRISPR/药物与阴性对照。
也可填写 `CRISPR_GENES`、`COMPOUNDS`、`EXPERIMENT`、`PLATE`，以及每个条件的重复数。
自动对比严格匹配 experiment、plate、扰动类型和 control-rules 中的额外字段；找不到对照时跳过并说明，不跨板补对照。
药物优先选有多个浓度的同板条件；CRISPR 优先展示同基因的不同 guide。自动选择只为探索，不代表扰动必然有显著表型。

In [ ]:
WELL_IDS = []                 # 例如 ['gene-077_3_L32', 'compound-001_10_AA12']；填写后优先使用。
CRISPR_GENES = []             # 留空自动选择最多两个基因。
COMPOUNDS = []                # 留空自动选择最多两个化合物。
EXPERIMENT = None             # 例如 'compound-001'。
PLATE = None                  # 例如 10。
REPLICATES_PER_CONDITION = 1
MAX_DOSES = 3
SHARED_CHANNEL_SCALE = True   # 每组每通道共用色阶，保留组内强度差异。

rules = json.loads(CONTROL_RULES_PATH.read_text(encoding='utf-8'))
metadata['is_negative_control'] = False
for kind, rule in rules.items():
    mask = metadata.perturbation_type.eq(kind)
    for field, values in rule['where'].items():
        mask &= metadata[field].isin([values] if isinstance(values, str) else values)
    if (metadata.loc[mask, 'is_query']).any() or metadata.loc[mask, 'well_type_label'].str.contains('positive', case=False).any():
        raise ValueError('阴性对照规则选中了 query，必须先修正规则')
    metadata.loc[mask, 'is_negative_control'] = True
display(metadata.groupby(['perturbation_type', 'is_negative_control']).size().rename('wells').to_frame())

pool = metadata.loc[metadata.is_query].copy()
if EXPERIMENT is not None:
    pool = pool.loc[pool.experiment_name.eq(EXPERIMENT)]
if PLATE is not None:
    pool = pool.loc[pool.plate.astype(int).eq(int(PLATE))]
display(pool[['well_id', 'experiment_name', 'plate', 'gene', 'treatment', 'concentration']].head(30))

def matched_controls(row):
    controls = metadata.loc[metadata.is_negative_control & metadata.perturbation_type.eq(row.perturbation_type)]
    fields = ['experiment_name', 'plate', *rules[row.perturbation_type].get('match_fields', [])]
    for field in dict.fromkeys(fields):
        if not str(row[field]).strip():
            return controls.iloc[:0]
        controls = controls.loc[controls[field].eq(row[field])]
    return controls.sort_values('well_id').head(REPLICATES_PER_CONDITION)

comparison_groups = []
if WELL_IDS:
    unknown = set(WELL_IDS) - set(metadata.well_id)
    if unknown:
        raise ValueError(f'未知 well_id: {sorted(unknown)}')
    comparison_groups.append(('Selected wells (manual selection)', list(dict.fromkeys(WELL_IDS))))
else:
    for kind, field, requested in [('CRISPR', 'gene', CRISPR_GENES), ('COMPOUND', 'treatment', COMPOUNDS)]:
        if kind not in rules:
            print('No explicit negative control rule:', kind)
            continue
        queries = pool.loc[pool.perturbation_type.eq(kind)]
        labels = requested or sorted(queries[field].unique())
        if not requested and kind == 'COMPOUND':
            dose_counts = queries.groupby(['treatment', 'experiment_name', 'plate']).concentration_um.nunique()
            labels = dose_counts.groupby(level=0).max().sort_values(ascending=False, kind='stable').index.tolist()
        added = 0
        for label in labels:
            candidates = []
            context_fields = list(dict.fromkeys(['experiment_name', 'plate', *rules[kind].get('match_fields', [])]))
            for _, context in queries.loc[queries[field].eq(label)].groupby(context_fields, sort=True):
                controls = matched_controls(context.iloc[0])
                if controls.empty:
                    continue
                condition = 'treatment' if kind == 'CRISPR' else 'concentration_um'
                selected = context.sort_values([condition, 'well_id']).groupby(condition, sort=True).head(REPLICATES_PER_CONDITION)
                conditions = sorted(selected[condition].dropna().unique())[:(2 if kind == 'CRISPR' else MAX_DOSES)]
                selected = selected.loc[selected[condition].isin(conditions)]
                if not selected.empty:
                    candidates.append((len(conditions), controls.well_id.tolist() + selected.well_id.tolist()))
            if not candidates:
                if requested:
                    print('No strictly matched comparison:', kind, label)
                continue
            _, ids = max(candidates, key=lambda item: item[0])
            comparison_groups.append((f'{kind}: {label}', ids))
            added += 1
            if not requested and added == 2:
                break
if not comparison_groups:
    print('无匹配组；请核实对照规则、筛选条件，或填写 WELL_IDS。')
for title, ids in comparison_groups:
    print(title)
    display(metadata.set_index('well_id').loc[ids, ['gene', 'treatment', 'concentration',
            'experiment_name', 'plate', 'well_type_label', 'is_negative_control']])

## 4. 原始六通道及伪彩对比

先扫描 `__key__` 列定位所选孔位，只加载命中的 row group 并解码所选六通道图像；不解码整个数据集。
同组每通道默认使用公共 1%–99.5% 分位数色阶；独立拉伸可能掩盖亮度变化。
伪彩为 ch6/ch3/ch1 → R/G/B，仅供展示，不定义真实染料颜色，也不改变模型的 `[0,1]` 输入。
修改 well 选择后重新运行选择与展示单元格即可。

In [ ]:
wanted = list(dict.fromkeys(well for _, ids in comparison_groups for well in ids))
selected_wells = read_selected_wells(shards, wanted) if wanted else {}
images = {}
for well_id, well in selected_wells.items():
    channels = []
    for raw in well.channels:
        with Image.open(BytesIO(raw)) as image:
            channels.append(np.array(image))
    images[well_id] = np.stack(channels)

def display_scale(array, low, high):
    return np.clip((array.astype(np.float32) - low) / max(float(high - low), 1.), 0, 1)

for title, ids in comparison_groups:
    stack = np.stack([images[well_id] for well_id in ids])
    limits = [np.percentile(stack[:, channel], [1, 99.5]) for channel in range(6)]
    fig, axes = plt.subplots(len(ids), 7, figsize=(20, 2.6 * len(ids)), squeeze=False)
    for i, well_id in enumerate(ids):
        row = metadata.loc[metadata.well_id.eq(well_id)].iloc[0]
        scaled = []
        for channel in range(6):
            array = images[well_id][channel]
            low, high = limits[channel] if SHARED_CHANNEL_SCALE else np.percentile(array, [1, 99.5])
            scaled.append(display_scale(array, low, high))
            axes[i, channel].imshow(scaled[-1], cmap='gray', vmin=0, vmax=1)
        axes[i, 6].imshow(np.stack([scaled[5], scaled[2], scaled[0]], axis=-1))
        label = 'NEGATIVE CONTROL' if row.is_negative_control else f'{row.treatment} / {row.concentration} uM'
        axes[i, 0].set_ylabel(f'{well_id}\n{label}', fontsize=8)
        for ax in axes[i]:
            ax.set_xticks([])
            ax.set_yticks([])
    for ax, label in zip(axes[0], ['ch1', 'ch2', 'ch3', 'ch4', 'ch5', 'ch6', 'Pseudo-RGB']):
        ax.set_title(label)
    fig.suptitle(title + (' | shared channel scales within this group' if SHARED_CHANNEL_SCALE else ' | independent scales'))
    plt.tight_layout()
    plt.show()

## 5. 可选 MDS 检查：仅查看清单和一个完整图像 batch

In [ ]:
if MDS_ROOT_TEXT:
    from phenostate.data.inference_loader import local_well_batches
    root = Path(MDS_ROOT_TEXT)
    manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
    display(pd.Series(manifest['counts'], name='wells').to_frame())
    split = next(name for name, count in manifest['counts'].items() if count > 0)
    with local_well_batches(root, split, batch_size=1, num_workers=0,
                            expected_wells=manifest['counts'][split]) as loader:
        batch = next(loader)
        print(split, batch['well_id'], tuple(batch['image'].shape))
else:
    print('RXRX3_MDS_ROOT 未设置，跳过 MDS 检查。')